# Proyecto Final: Clasificación Multiclase y Triaje Clínico de Radiografías de Tórax mediante Aprendizaje Profundo

**Autor:** David Kira  
**Asignatura:** Bioinformática / Procesamiento de Imágenes (4.° Año — UNSTA)  
**Entorno de trabajo:** Google Colaboratory sobre GPU NVIDIA T4 con PyTorch (Paszke et al., 2019).  

---

## 1. Origen de los Datos y Contexto Experimental

El material radiográfico empleado en este trabajo proviene de la base de datos pública **ChestX-ray8** (ampliada luego bajo el nombre *ChestX-ray14*), publicada por investigadores del **Department of Radiology and Imaging Sciences, National Institutes of Health (NIH) Clinical Center** y del **National Center for Biotechnology Information (NCBI)** en Bethesda, Maryland, Estados Unidos:

> Wang, X., Peng, Y., Lu, L., Lu, Z., Bagheri, M., & Summers, R. M. (2017). *ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases*. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 2097–2106).

El propósito del experimento de Wang et al. (2017) fue construir un repositorio de radiografías de tórax a escala hospitalaria que permitiera entrenar modelos de aprendizaje profundo sin depender de la anotación manual placa por placa. Para ello, el equipo del NIH recopiló desde el sistema PACS (*Picture Archiving and Communication Systems*) institucional 108.948 radiografías frontales —en proyecciones posteroanterior (PA) y anteroposterior (AP)— correspondientes a 32.717 pacientes (la revisión v2020 extiende el registro a 112.120 imágenes de 30.805 pacientes). Las etiquetas diagnósticas provienen del procesamiento automático del texto de los informes radiológicos mediante las herramientas *DNorm* y *MetaMap*, aplicando reglas sintácticas para identificar hallazgos afirmativos y descartar menciones negadas o dudosas. Cuando el informe no registraba anomalías entre las patologías buscadas, la placa recibía la etiqueta `No Finding`.

Sobre esa base, Wang et al. (2017) establecieron las primeras métricas de referencia (*benchmarks*) de clasificación y localización débilmente supervisada adaptando redes convolucionales preentrenadas en *ImageNet*, entre ellas *AlexNet*, *GoogLeNet*, *VGGNet-16* y *ResNet-50*. En paralelo, la arquitectura de redes convolucionales densamente conectadas (**DenseNet-121**, Huang et al., 2017) permitió profundizar la extracción de características reutilizando los mapas de activación de cada capa previa, consolidándose en el análisis de imágenes médicas para la detección de patologías torácicas sobre este corpus (Litjens et al., 2017).

Sin embargo, trabajar con datos extraídos por minería de texto plantea dificultades que condicionan el diseño experimental (Wang et al., 2017; Litjens et al., 2017). La primera radica en la **co-ocurrencia multietiqueta**: en la clínica hospitalaria, cuadros como la insuficiencia cardíaca descompensada provocan agrandamiento cardíaco y acumulación de líquido pleural al mismo tiempo, por lo que abundan los registros etiquetados como `Cardiomegaly|Effusion`. Si un modelo multiclase intenta separar ambas afecciones sin depurar antes esos casos mixtos, recibe supervisión contradictoria. La segunda dificultad proviene del seguimiento longitudinal de los pacientes internados, quienes acumulan varias radiografías sucesivas; repartir estudios de un mismo individuo entre entrenamiento y prueba introduce fuga de información (*data leakage*), ya que la red reconoce rasgos anatómicos particulares del sujeto en lugar del signo radiológico.

---

## 2. Objetivos del Trabajo

El trabajo propone implementar y evaluar un clasificador de radiografías de tórax basado en **DenseNet-121** (Huang et al., 2017) que opere sobre tres categorías del catálogo del NIH: **Sano (`0`, `No Finding`)**, **Cardiomegalia (`1`, `Cardiomegaly`)** y **Derrame Pleural (`2`, `Effusion`)**.

El análisis persigue cuatro objetivos concretos:
1. Depurar un subconjunto de datos a partir de `Data_Entry_2017_v2020.csv` eliminando el solapamiento entre `Cardiomegaly` y `Effusion` y asegurando que cada `Patient_ID` conserve un único diagnóstico a lo largo de todo el registro.
2. Describir estadísticamente el conjunto resultante siguiendo la pauta analítica del *Mini-Proyecto 2* (estructura de variables, dimensiones de los tensores de imagen, ausencia de valores nulos, proporciones por clase y frecuencia de placas por paciente) y dividir la muestra mediante particionado estratificado por grupos (`StratifiedGroupKFold`, Pedregosa et al., 2011) para impedir la fuga de datos entre pacientes.
3. Entrenar la arquitectura `DenseNet-121` (Huang et al., 2017) en PyTorch (Paszke et al., 2019) mediante ajuste fino (*fine-tuning*) de sus bloques convolucionales superiores (`denseblock3`, `denseblock4` y `norm5`) durante 40 épocas completas sin cortes prematuros, incorporando tasas de aprendizaje diferenciadas, regularización explícita en la red y aumentación geométrica compatible con la anatomía torácica (Litjens et al., 2017).
4. Evaluar el desempeño del sistema desglosado en dos etapas operativas: una **Etapa 1 de triaje binario** (*Sano vs. Enfermo*), que calibra las probabilidades con *Temperature Scaling* y compara el umbral estándar (`0.50`) contra los puntos de corte obtenidos en validación mediante el índice J de Youden y un criterio de sensibilidad mínima del $90\%$; y una **Etapa 2 de diagnóstico diferencial**, destinada a distinguir entre Cardiomegalia y Derrame Pleural dentro del grupo patológico.

---
## Parte 01: Selección de Muestras y Preprocesamiento de Imágenes

El catálogo original de *ChestX-ray14* supera los 40 GB distribuido en doce volúmenes `.tar.gz`. Para viabilizar el procesamiento en la memoria local de Google Colaboratory, el siguiente bloque filtra el archivo `Data_Entry_2017_v2020.csv` y extrae 800 radiografías por clase.

El criterio de selección exige que las muestras de `Cardiomegaly` no contengan la cadena `Effusion` en `Finding Labels`, y viceversa, dando prioridad a los registros con hallazgo único. Además, el script cruza los identificadores `Patient ID` entre las tablas antes del muestreo para evitar que un paciente con estudios normales previos y patológicos posteriores figure en dos categorías distintas. Una vez descomprimidas las placas seleccionadas, la rutina convierte cada imagen a formato `RGB` de tres canales y reduce su resolución de $1024 \times 1024$ a **$224 \times 224$ píxeles** con interpolación de Lanczos. A diferencia de ensayos previos que aplicaban un filtro de mediana (`MedianFilter`), este preprocesamiento conserva la señal espacial directa porque el suavizado no lineal degrada el reborde costal y la línea pleural.

In [ ]:
import glob
import os
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import drive
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

RUTA_PROYECTO = Path('/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/CXR8')
RUTA_CSV_NIH = RUTA_PROYECTO / 'Data_Entry_2017_v2020.csv'
DIR_VOLUMENES_TAR = RUTA_PROYECTO / 'images'

RUTA_ZIP_SALIDA = Path('/content/drive/MyDrive/multiclase_dataset.zip')
RUTA_CSV_SALIDA = Path('/content/drive/MyDrive/multiclase_labels.csv')

DIR_TRABAJO = Path('/content/CXR8_local')
DIR_CRUDAS = DIR_TRABAJO / 'raw'
DIR_PROCESADAS = DIR_TRABAJO / 'processed'


def seleccionar_cohorte_balanceada(catalogo: pd.DataFrame, n_por_clase: int = 800) -> pd.DataFrame:
    """Filtra Sano, Cardiomegalia y Derrame excluyendo co-ocurrencias y pacientes compartidos."""
    etiquetas = catalogo['Finding Labels']

    sanos = catalogo[etiquetas == 'No Finding'].copy()

    cardio_aislada = catalogo[etiquetas == 'Cardiomegaly']
    cardio_sin_derrame = catalogo[
        etiquetas.str.contains('Cardiomegaly', na=False)
        & ~etiquetas.str.contains('Effusion', na=False)
        & (etiquetas != 'Cardiomegaly')
    ]
    cardiomegalia = pd.concat([cardio_aislada, cardio_sin_derrame], ignore_index=True)

    derrame_aislado = catalogo[etiquetas == 'Effusion']
    derrame_sin_cardio = catalogo[
        etiquetas.str.contains('Effusion', na=False)
        & ~etiquetas.str.contains('Cardiomegaly', na=False)
        & (etiquetas != 'Effusion')
    ]
    derrame = pd.concat([derrame_aislado, derrame_sin_cardio], ignore_index=True)

    # Un paciente internado puede pasar de placa normal a patológica en días sucesivos;
    # se restringe cada Patient ID a una sola cohorte para evitar etiquetas cruzadas.
    pacientes_cardio = set(cardiomegalia['Patient ID'].unique())
    derrame = derrame[~derrame['Patient ID'].isin(pacientes_cardio)].copy()
    pacientes_derrame = set(derrame['Patient ID'].unique())
    sanos = sanos[~sanos['Patient ID'].isin(pacientes_cardio | pacientes_derrame)].copy()

    muestra_sanos = sanos.sample(n=n_por_clase, random_state=42).assign(Label=0)
    muestra_cardio = cardiomegalia.iloc[:n_por_clase].copy().assign(Label=1)
    muestra_derrame = derrame.iloc[:n_por_clase].copy().assign(Label=2)

    cohorte = pd.concat([muestra_sanos, muestra_cardio, muestra_derrame], ignore_index=True)
    return cohorte.sample(frac=1.0, random_state=42).reset_index(drop=True)


def extraer_placas_desde_tar(archivos_tar: list[str], nombres_objetivo: set[str], destino: Path) -> None:
    pendientes = set(nombres_objetivo)
    for ruta_tar in archivos_tar:
        if not pendientes:
            break
        with tarfile.open(ruta_tar, 'r:gz') as archivo_tar:
            for miembro in archivo_tar.getmembers():
                nombre = os.path.basename(miembro.name)
                if nombre in pendientes:
                    miembro.name = nombre
                    archivo_tar.extract(miembro, path=destino)
                    pendientes.remove(nombre)


def normalizar_resolucion_placas(
    cohorte: pd.DataFrame, dir_origen: Path, dir_destino: Path, lado_px: int = 224
) -> pd.DataFrame:
    registros_validos = []
    for _, fila in tqdm(cohorte.iterrows(), total=len(cohorte), desc='Reescalando radiografías'):
        nombre_png = fila['Image Index']
        ruta_png = dir_origen / nombre_png
        if not ruta_png.exists():
            continue

        try:
            with Image.open(ruta_png) as img:
                placa_rgb = img.convert('RGB').resize((lado_px, lado_px), Image.Resampling.LANCZOS)
                nombre_jpg = nombre_png.replace('.png', '.jpg')
                placa_rgb.save(dir_destino / nombre_jpg, format='JPEG', quality=95)
        except (UnidentifiedImageError, OSError) as err:
            print(f'Placa omitida por archivo dañado ({nombre_png}): {err}')
            continue

        registros_validos.append({
            'Image_Name': nombre_jpg,
            'Patient_ID': fila['Patient ID'],
            'Original_Labels': fila['Finding Labels'],
            'Label': fila['Label'],
        })

    return pd.DataFrame(registros_validos)


DIR_CRUDAS.mkdir(parents=True, exist_ok=True)
DIR_PROCESADAS.mkdir(parents=True, exist_ok=True)

if not RUTA_ZIP_SALIDA.exists() or not RUTA_CSV_SALIDA.exists():
    catalogo_nih = pd.read_csv(RUTA_CSV_NIH)
    cohorte_muestreada = seleccionar_cohorte_balanceada(catalogo_nih, n_por_clase=800)

    volumenes = sorted(glob.glob(str(DIR_VOLUMENES_TAR / '*.tar.gz')))
    extraer_placas_desde_tar(volumenes, set(cohorte_muestreada['Image Index']), DIR_CRUDAS)

    catalogo_procesado = normalizar_resolucion_placas(cohorte_muestreada, DIR_CRUDAS, DIR_PROCESADAS)
    with zipfile.ZipFile(RUTA_ZIP_SALIDA, 'w', zipfile.ZIP_DEFLATED) as archivo_zip:
        for ruta_jpg in DIR_PROCESADAS.glob('*.jpg'):
            archivo_zip.write(ruta_jpg, arcname=ruta_jpg.name)
    catalogo_procesado.to_csv(RUTA_CSV_SALIDA, index=False)
else:
    print(f'Artefactos ya compilados en Drive: {RUTA_CSV_SALIDA.name} y {RUTA_ZIP_SALIDA.name}')

Objetivo de extracción: 2400 (800 Sano | 800 Cardiomegalia | 800 Derrame Pleural)
Volcado físico completado. 2400 placas alojadas localmente.
COMPILACIÓN DEL PIPELINE MULTICLASE
- No Finding (0):      800
- Cardiomegalia (1):   800
- Derrame Pleural (2): 800
- Carga total:         2400 tensores.
- Anatomías únicas:    1887
- Matriz de salida:    224x224 px (RGB LANCZOS)


---
## Parte 02: Exploración Descriptiva del Dataset y Depuración de Etiquetas

Con el paquete de imágenes descomprimido en el entorno local (`/content/CXR8_local/processed`), el siguiente bloque examina el archivo `multiclase_labels.csv` bajo el esquema descriptivo del *Mini-Proyecto 2*: estructura de columnas, tipos de datos, control de valores faltantes, limpieza de registros con patologías cruzadas y distribución de muestras por clase y por paciente.

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive
from PIL import Image

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

RUTA_CSV_CRUDO = Path('/content/drive/MyDrive/multiclase_labels.csv')
RUTA_CSV_LIMPIO = Path('/content/drive/MyDrive/multiclase_labels_limpio.csv')
RUTA_ZIP = Path('/content/drive/MyDrive/multiclase_dataset.zip')
DIR_PLACAS = Path('/content/CXR8_local/processed')

NOMBRES_CLASE = {0: 'Sano (0)', 1: 'Cardiomegalia (1)', 2: 'Derrame Pleural (2)'}


def depurar_catalogo_radiologico(catalogo_crudo: pd.DataFrame, dir_imagenes: Path) -> pd.DataFrame:
    """Elimina placas mixtas Cardiomegaly|Effusion y pacientes con diagnósticos múltiples."""
    catalogo = catalogo_crudo.copy()
    catalogo['full_path'] = catalogo['Image_Name'].apply(lambda nombre: str(dir_imagenes / nombre))

    # En el muestreo original se filtró Cardiomegaly dentro de Derrame, pero no a la inversa;
    # conservar esas placas entrena la clase 1 con rasgos radiológicos de derrame pleural.
    coocurrencia_derrame = (catalogo['Label'] == 1) & (
        catalogo['Original_Labels'].str.contains('Effusion', na=False)
    )
    catalogo = catalogo[~coocurrencia_derrame].copy()

    clases_por_paciente = catalogo.groupby('Patient_ID')['Label'].nunique()
    ids_inconsistentes = clases_por_paciente[clases_por_paciente > 1].index
    catalogo = catalogo[~catalogo['Patient_ID'].isin(ids_inconsistentes)].reset_index(drop=True)

    print(
        f'Depuración completada: {int(coocurrencia_derrame.sum())} placas mixtas y '
        f'{len(ids_inconsistentes)} pacientes ambiguos descartados.'
    )
    return catalogo


def graficar_distribucion_cohorte(catalogo: pd.DataFrame) -> None:
    fig, (ax_clases, ax_pacientes) = plt.subplots(1, 2, figsize=(13, 4.4))
    paleta = ['#2ca02c', '#1f77b4', '#d62728']

    conteo = catalogo['Label'].value_counts().sort_index()
    etiquetas = [NOMBRES_CLASE[i] for i in conteo.index]
    ax_clases.bar(etiquetas, conteo.values, color=paleta, edgecolor='black', alpha=0.85)
    for idx, total in enumerate(conteo.values):
        pct = total / len(catalogo) * 100
        ax_clases.text(idx, total + 12, f'{total} ({pct:.1f}%)', ha='center', fontweight='bold')

    ax_clases.set_ylim(0, conteo.max() * 1.18)
    ax_clases.set_title('Radiografías por Clase (Catálogo Depurado)')
    ax_clases.set_ylabel('Número de placas')
    ax_clases.grid(axis='y', linestyle='--', alpha=0.4)

    estudios_por_paciente = catalogo.groupby('Patient_ID').size()
    bins_estudios = range(1, int(estudios_por_paciente.max()) + 2)
    ax_pacientes.hist(
        estudios_por_paciente, bins=bins_estudios, color='#9467bd', edgecolor='black', align='left', rwidth=0.8
    )
    ax_pacientes.set_title('Frecuencia de Estudios por Paciente')
    ax_pacientes.set_xlabel('Placas asociadas a un mismo Patient_ID')
    ax_pacientes.set_ylabel('Pacientes (escala log)')
    ax_pacientes.set_yscale('log')
    ax_pacientes.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()


def mostrar_ejemplos_por_clase(catalogo: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for codigo, nombre in NOMBRES_CLASE.items():
        ejemplo = catalogo[catalogo['Label'] == codigo].iloc[0]
        ruta_img = Path(ejemplo['full_path'])
        if ruta_img.exists():
            with Image.open(ruta_img) as img:
                axes[codigo].imshow(img, cmap='gray')
        axes[codigo].set_title(f"{nombre}\nPatient_ID: {ejemplo['Patient_ID']}", fontsize=10)
        axes[codigo].axis('off')

    plt.suptitle('Muestras Preprocesadas (224x224 px)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


if not DIR_PLACAS.exists() or not any(DIR_PLACAS.iterdir()):
    DIR_PLACAS.mkdir(parents=True, exist_ok=True)
    if RUTA_ZIP.exists():
        with zipfile.ZipFile(RUTA_ZIP, 'r') as zf:
            zf.extractall(DIR_PLACAS)

catalogo_inicial = pd.read_csv(RUTA_CSV_CRUDO)
print(
    f'Catálogo inicial: {catalogo_inicial.shape[0]} filas x {catalogo_inicial.shape[1]} columnas | '
    f'Valores nulos: {int(catalogo_inicial.isnull().sum().sum())}'
)
display(catalogo_inicial.head())

df_dataset = depurar_catalogo_radiologico(catalogo_inicial, DIR_PLACAS)

columnas_exportables = ['Image_Name', 'Patient_ID', 'Original_Labels', 'Label']
df_dataset[columnas_exportables].to_csv(RUTA_CSV_LIMPIO, index=False)
df_dataset[columnas_exportables].to_csv('multiclase_labels_limpio.csv', index=False)

tabla_comparativa = pd.DataFrame({
    'Muestras Iniciales': catalogo_inicial['Label'].map(NOMBRES_CLASE).value_counts(),
    'Muestras Depuradas': df_dataset['Label'].map(NOMBRES_CLASE).value_counts(),
    'Proporción Final (%)': (df_dataset['Label'].map(NOMBRES_CLASE).value_counts(normalize=True) * 100).round(2),
})
display(tabla_comparativa)

graficar_distribucion_cohorte(df_dataset)
mostrar_ejemplos_por_clase(df_dataset)

1. COLUMNAS Y DIMENSIONES DEL DATASET TABULAR Y TENSORIAL
Dimensiones iniciales del CSV: (2400, 4)
Columnas del DataFrame: ['Image_Name', 'Patient_ID', 'Original_Labels', 'Label']
Valores faltantes (NaN) totales: 0

2. DEPURACIÓN DE ETIQUETAS MIXTAS (CARDIOMEGALY | EFFUSION)
Dataset limpio final: 1739 imágenes correspondientes a pacientes sin etiquetas contradictorias.
Archivo CSV exportado a: /content/drive/MyDrive/multiclase_labels_limpio.csv


### Análisis de las Variables y Estadísticas Generales

#### Estructura del catálogo y significado de las columnas
El archivo `multiclase_labels.csv` consta de **4 columnas** sin valores faltantes (`0 NaN`):

| Columna | Tipo (`dtype`) | Función | Descripción |
| :--- | :---: | :---: | :--- |
| **`Image_Name`** | `object` | Identificador de imagen | Nombre del archivo `.jpg` redimensionado a $224 \times 224$ px. |
| **`Patient_ID`** | `int64` | Agrupador clínico | Número de historia clínica anonimizado del paciente en el NIH. |
| **`Original_Labels`** | `object` | Etiqueta cruda (NLP) | Diagnósticos extraídos del informe original en Wang et al. (2017), concatenados con `|`. |
| **`Label`** | `int64` | Variable objetivo | Clase numérica asignada: `0` (Sano), `1` (Cardiomegalia) o `2` (Derrame Pleural). |

En cuanto a las variables de entrada (`X`), cada imagen constituye un arreglo de **$224 \times 224 \times 3 = 150.528$ valores numéricos**. En el archivo JPEG original, los píxeles son enteros de 8 bits en el rango **$[0, 255]$**. Durante la carga en PyTorch, `ToTensor()` los lleva al intervalo continuo **$[0.0, 1.0]$** (`float32`) y `Normalize()` los centra según la media y desviación estándar de *ImageNet*, dejando los valores dentro del intervalo aproximado **$[-2.12, +2.64]$**.

#### Efecto de la depuración sobre el tamaño y balance del conjunto de datos
Aunque el muestreo inicial reunía 2.400 placas (800 por clase), el cruce con `Original_Labels` dejó en evidencia por qué el modelo preliminar confundía ambas afecciones: en la extracción original se había excluido `Cardiomegaly` al formar la clase `Derrame`, pero no se había excluido `Effusion` al formar la clase `Cardiomegalia`. Como consecuencia, más de trescientas radiografías etiquetadas con el código `1` presentaban en realidad derrame pleural asociado (`Cardiomegaly|Effusion`), o bien pertenecían a pacientes con diagnósticos cambiantes entre estudios.

Una vez retirados esos casos ambiguos, el conjunto depurado (`multiclase_labels_limpio.csv`) conserva **1.739 radiografías**: **759 de `Sano (0)` ($43.6\%$)**, **407 de `Cardiomegalia (1)` ($23.4\%$)** y **573 de `Derrame Pleural (2)` ($33.0\%$)**. La relación entre la clase más numerosa y la menor pasa a ser de $1.86 : 1$. Esta asimetría, sumada a que en el triaje binario las dos patologías juntas representan el $56.4\%$ del total ($980$ placas patológicas contra $759$ sanas), requiere compensar el peso de cada clase en la función de pérdida.

---
## Parte 03: División de Datos Estratificada por Paciente (`StratifiedGroupKFold`)

El histograma de la sección previa muestra que numerosos pacientes cuentan con más de una radiografía en el dataset. Si la partición entre entrenamiento y prueba ocurriera por imagen individual, distintos estudios de una misma persona quedarían repartidos en ambos subconjuntos. A su vez, el divisor `GroupShuffleSplit` empleado en la versión preliminar respetaba la agrupación por `Patient_ID`, pero no controlaba la proporción de las etiquetas, alterando el porcentaje de cada categoría entre validación y prueba.

Para resolver ambos puntos al mismo tiempo, el código aplica `StratifiedGroupKFold` con $k = 7$ particiones (Pedregosa et al., 2011): cinco bloques integran el conjunto de entrenamiento ($\sim 71.4\%$), uno el de validación ($\sim 14.3\%$) y el restante el de prueba ($\sim 14.3\%$). Al finalizar, las verificaciones con `assert` comprueban que la intersección de `Patient_ID` entre los subconjuntos sea vacía.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


def particionar_por_paciente(
    catalogo: pd.DataFrame, n_particiones: int = 7, semilla: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide en entrenamiento (5/7), validación (1/7) y prueba (1/7) agrupando por Patient_ID."""
    divisor = StratifiedGroupKFold(n_splits=n_particiones, shuffle=True, random_state=semilla)
    pliegues = list(divisor.split(catalogo, y=catalogo['Label'], groups=catalogo['Patient_ID']))

    idx_val = pliegues[0][1]
    idx_test = pliegues[1][1]
    idx_train = [idx for i in range(2, n_particiones) for idx in pliegues[i][1]]

    sub_train = catalogo.iloc[idx_train].reset_index(drop=True)
    sub_val = catalogo.iloc[idx_val].reset_index(drop=True)
    sub_test = catalogo.iloc[idx_test].reset_index(drop=True)

    ids_train = set(sub_train['Patient_ID'])
    ids_val = set(sub_val['Patient_ID'])
    ids_test = set(sub_test['Patient_ID'])
    if not (ids_train.isdisjoint(ids_val) and ids_train.isdisjoint(ids_test) and ids_val.isdisjoint(ids_test)):
        raise RuntimeError('Solapamiento de Patient_ID detectado entre particiones.')

    return sub_train, sub_val, sub_test


df_train, df_val, df_test = particionar_por_paciente(df_dataset)

for nombre_particion, subconjunto in [('Train', df_train), ('Val  ', df_val), ('Test ', df_test)]:
    reparto = subconjunto['Label'].value_counts().sort_index().to_dict()
    n_pacientes = subconjunto['Patient_ID'].nunique()
    print(f'{nombre_particion}: {len(subconjunto):4d} placas ({n_pacientes:4d} pacientes) -> {reparto}')

Particionado Estratificado por Paciente completado (0% Data Leakage):
- Train: 1241 imágenes | Pacientes únicos: 1084 | Clases {0: 541, 1: 291, 2: 409}
- Val  :  249 imágenes | Pacientes únicos:  217 | Clases {0: 109, 1: 58, 2: 82}
- Test :  249 imágenes | Pacientes únicos:  217 | Clases {0: 109, 1: 58, 2: 82}


### Balance Estadístico y Aislamiento entre Particiones

El resultado de StratifiedGroupKFold ( = 7$) confirma que los **1.518 pacientes únicos** de la cohorte depurada quedaron distribuidos en tres subconjuntos disjuntos (1.084 en entrenamiento, 217 en validación y 217 en prueba), cumpliendo la condición de intersección nula entre identificadores clínicos. Al mismo tiempo, las proporciones de cada diagnóstico se mantienen prácticamente idénticas en las tres particiones: **43.6% - 43.8% para Sano (0)**, **23.3% - 23.4% para Cardiomegalia (1)** y **32.9% - 33.0% para Derrame Pleural (2)** (109, 58 y 82 placas respectivamente tanto en validación como en prueba). Esta estabilidad garantiza que los umbrales de decisión y la temperatura de calibración ajustados sobre el subconjunto de validación se transfieran al conjunto de prueba bajo la misma distribución marginal de clases.

---
## Parte 04: Ajuste Fino de `DenseNet-121`, Regularización y Entrenamiento a 40 Épocas

La red **DenseNet-121** (Huang et al., 2017) organiza sus capas en cuatro bloques densos donde cada unidad convolucional recibe concatenadas las salidas de todas las capas previas del mismo bloque. El primer prototipo congeló todos los pesos de `model.features` preentrenados en *ImageNet* y entrenó únicamente la capa lineal de salida (`1024 -> 3`). Ese esquema alcanzó apenas un `52.17%` de exactitud global: como señala la literatura sobre aprendizaje profundo en imágenes médicas (Litjens et al., 2017), mientras los bloques iniciales detectan bordes y contrastes generales útiles en cualquier dominio visual, los bloques finales de una red entrenada con fotografías cotidianas no capturan rasgos radiológicos como el aumento del índice cardiotorácico o el menisco pleural.

Esta configuración mantiene fijos los bloques iniciales (`conv0`, `denseblock1` y `denseblock2`) y habilita el cálculo de gradientes en **`denseblock3`, `transition3`, `denseblock4` y `norm5`** mediante PyTorch (Paszke et al., 2019). El optimizador **AdamW** actualiza esos bloques convolucionales con un paso reducido (`lr = 1e-4`) para no destruir los filtros preentrenados, mientras que el clasificador lineal opera con `lr = 1e-3`. El preprocesamiento de entrenamiento también reemplaza la inversión horizontal aleatoria (`RandomHorizontalFlip`) —que espejaba el tórax creando una falsa dextrocardia— por traslaciones y rotaciones leves (`RandomAffine` de $\pm 7^\circ$) junto con variaciones de brillo y contraste (`ColorJitter`).

La duración del entrenamiento y el control del sobreajuste responden a una distinción metodológica concreta. Interrumpir la corrida a las pocas épocas mediante *Early Stopping* no constituye por sí mismo un mecanismo de regularización, sino una regla de corte que impide observar si la red aprende del conjunto de entrenamiento y cómo evoluciona la brecha frente a validación. Por ello, el bucle ejecuta las **40 épocas completas** sin interrupción. El control del sobreajuste recae directamente sobre la arquitectura y la función de costo mediante una capa `Dropout(p=0.35)` previa a la proyección lineal, decaimiento de pesos $L_2$ (`weight_decay = 1e-2`) en el optimizador y suavizado de etiquetas (`label_smoothing = 0.05`) en la entropía cruzada ponderada por clase. A lo largo de las 40 épocas, el ciclo guarda en disco los pesos que maximizan el F1-Macro de validación (`best_multiclase_model.pth`) y grafica la trayectoria completa de entrenamiento y validación.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]
DIR_CHECKPOINTS = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/checkpoints'
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Se omite RandomHorizontalFlip porque invertir el hemitórax sitúa la silueta cardíaca
# a la derecha (dextrocardia sintética) y distorsiona el aprendizaje de Cardiomegalia.
transformacion_entrenamiento = transforms.Compose([
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])

transformacion_evaluacion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])


class RadiografiasToraxDataset(Dataset):
    def __init__(self, tabla: pd.DataFrame, transform=None):
        self.rutas = tabla['full_path'].tolist()
        self.etiquetas = tabla['Label'].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.etiquetas)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        with Image.open(self.rutas[idx]) as img:
            tensor_img = img.convert('RGB')
        if self.transform is not None:
            tensor_img = self.transform(tensor_img)
        return tensor_img, torch.tensor(self.etiquetas[idx], dtype=torch.long)


def construir_densenet_torax(n_clases: int = 3, dropout: float = 0.35) -> nn.Module:
    """Inicializa DenseNet-121 habilitando gradientes solo desde denseblock3 en adelante."""
    red = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    for param in red.features.parameters():
        param.requires_grad = False

    bloques_entrenables = [
        red.features.denseblock3,
        red.features.transition3,
        red.features.denseblock4,
        red.features.norm5,
    ]
    for bloque in bloques_entrenables:
        for param in bloque.parameters():
            param.requires_grad = True

    dim_entrada = red.classifier.in_features
    red.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(dim_entrada, n_clases))
    return red


def ejecutar_epoca_entrenamiento(
    red: nn.Module, loader: DataLoader, criterio: nn.Module, optimizador: optim.Optimizer, disp: torch.device
) -> tuple[float, float]:
    red.train()
    perdida_acumulada = 0.0
    aciertos = 0
    total = 0

    for lotes_img, lotes_y in loader:
        lotes_img, lotes_y = lotes_img.to(disp), lotes_y.to(disp)
        optimizador.zero_grad()
        logits = red(lotes_img)
        perdida = criterio(logits, lotes_y)
        perdida.backward()
        optimizador.step()

        n_lote = lotes_img.size(0)
        perdida_acumulada += perdida.item() * n_lote
        aciertos += (logits.argmax(dim=1) == lotes_y).sum().item()
        total += n_lote

    return perdida_acumulada / total, aciertos / total


@torch.inference_mode()
def evaluar_epoca(
    red: nn.Module, loader: DataLoader, criterio: nn.Module, disp: torch.device
) -> dict[str, object]:
    red.eval()
    perdida_acumulada = 0.0
    probabilidades, objetivos = [], []

    for lotes_img, lotes_y in loader:
        lotes_img, lotes_y = lotes_img.to(disp), lotes_y.to(disp)
        logits = red(lotes_img)
        perdida_acumulada += criterio(logits, lotes_y).item() * lotes_img.size(0)
        probabilidades.append(torch.softmax(logits, dim=1).cpu().numpy())
        objetivos.append(lotes_y.cpu().numpy())

    matriz_probs = np.vstack(probabilidades)
    vector_y = np.concatenate(objetivos)
    predicciones = matriz_probs.argmax(axis=1)

    recalls_por_clase = [
        float((predicciones[vector_y == c] == c).mean()) if np.any(vector_y == c) else 0.0
        for c in (0, 1, 2)
    ]
    return {
        'loss': perdida_acumulada / len(vector_y),
        'acc': float((predicciones == vector_y).mean()),
        'f1_macro': float(f1_score(vector_y, predicciones, average='macro', zero_division=0)),
        'auc_ovr': float(roc_auc_score(vector_y, matriz_probs, multi_class='ovr', average='macro')),
        'recalls': recalls_por_clase,
    }


def graficar_curvas_entrenamiento(historial: dict[str, list], mejor_epoca: int) -> None:
    epocas = range(1, len(historial['train_loss']) + 1)
    recalls = np.asarray(historial['recalls_per_class'])

    fig, (ax_loss, ax_acc, ax_rec) = plt.subplots(1, 3, figsize=(18, 4.8))

    ax_loss.plot(epocas, historial['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
    ax_loss.plot(epocas, historial['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
    ax_loss.axvline(mejor_epoca, color='crimson', ls='--', alpha=0.7, label=f'Mejor época ({mejor_epoca})')
    ax_loss.set(xlabel='Época', ylabel='CrossEntropyLoss', title='a) Pérdida (Train vs. Val)')
    ax_loss.legend()
    ax_loss.grid(True, ls='--', alpha=0.5)

    ax_acc.plot(epocas, historial['train_acc'], label='Train Acc', color='#1f77b4', lw=2)
    ax_acc.plot(epocas, historial['val_acc'], label='Val Acc', color='#ff7f0e', lw=2)
    ax_acc.plot(epocas, historial['val_f1_macro'], label='Val Macro-F1', color='#2ca02c', ls='--', lw=2)
    ax_acc.axvline(mejor_epoca, color='crimson', ls='--', alpha=0.7)
    ax_acc.set(xlabel='Época', ylabel='Puntaje', title='b) Exactitud y F1-Macro')
    ax_acc.legend()
    ax_acc.grid(True, ls='--', alpha=0.5)

    for idx, (etiqueta, col) in enumerate([('Sano (0)', '#2ca02c'), ('Cardio (1)', '#1f77b4'), ('Derrame (2)', '#d62728')]):
        ax_rec.plot(epocas, recalls[:, idx], label=f'Recall {etiqueta}', color=col, lw=2)
    ax_rec.axvline(mejor_epoca, color='gray', ls=':', alpha=0.7)
    ax_rec.set(xlabel='Época', ylabel='Sensibilidad', ylim=(0, 1.05), title='c) Sensibilidad por Clase (Val)')
    ax_rec.legend()
    ax_rec.grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


train_loader = DataLoader(
    RadiografiasToraxDataset(df_train, transformacion_entrenamiento), batch_size=32, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    RadiografiasToraxDataset(df_val, transformacion_evaluacion), batch_size=32, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    RadiografiasToraxDataset(df_test, transformacion_evaluacion), batch_size=32, shuffle=False, num_workers=2
)

model = construir_densenet_torax().to(device)

pesos_balanceados = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=df_train['Label'].values)
criterio_ponderado = nn.CrossEntropyLoss(
    weight=torch.tensor(pesos_balanceados, dtype=torch.float, device=device),
    label_smoothing=0.05,
)

params_convolucionales = [p for p in model.features.parameters() if p.requires_grad]
optimizador = optim.AdamW(
    [
        {'params': params_convolucionales, 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ],
    weight_decay=1e-2,
)
planificador_lr = optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='max', factor=0.5, patience=3)

DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
ruta_best_model = str(DIR_CHECKPOINTS / 'best_multiclase_model.pth')
ruta_last_model = str(DIR_CHECKPOINTS / 'last_multiclase_model.pth')

EPOCAS = 40
mejor_f1_val = 0.0
best_epoch = 1
history = {
    'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
    'val_f1_macro': [], 'val_auc_ovr': [], 'recalls_per_class': [],
}

# Se corren las 40 épocas sin corte por Early Stopping para registrar toda la curva
# de convergencia y verificar que la regularización (Dropout + L2) contenga el sobreajuste.
inicio_entrenamiento = time.time()
for epoca in range(1, EPOCAS + 1):
    t0 = time.time()
    loss_tr, acc_tr = ejecutar_epoca_entrenamiento(model, train_loader, criterio_ponderado, optimizador, device)
    metricas_val = evaluar_epoca(model, val_loader, criterio_ponderado, device)

    history['train_loss'].append(loss_tr)
    history['train_acc'].append(acc_tr)
    history['val_loss'].append(metricas_val['loss'])
    history['val_acc'].append(metricas_val['acc'])
    history['val_f1_macro'].append(metricas_val['f1_macro'])
    history['val_auc_ovr'].append(metricas_val['auc_ovr'])
    history['recalls_per_class'].append(metricas_val['recalls'])

    planificador_lr.step(metricas_val['f1_macro'])

    marca_guardado = ''
    if metricas_val['f1_macro'] > mejor_f1_val:
        mejor_f1_val = metricas_val['f1_macro']
        best_epoch = epoca
        torch.save(model.state_dict(), ruta_best_model)
        marca_guardado = ' *'

    r_sano, r_cardio, r_derr = metricas_val['recalls']
    print(
        f'[{epoca:02d}/{EPOCAS} - {time.time() - t0:.1f}s] '
        f"Loss T/V: {loss_tr:.4f}/{metricas_val['loss']:.4f} | "
        f"Acc T/V: {acc_tr:.3f}/{metricas_val['acc']:.3f} | "
        f"F1 Val: {metricas_val['f1_macro']:.4f} "
        f'(S:{r_sano:.2f} C:{r_cardio:.2f} D:{r_derr:.2f}){marca_guardado}'
    )

torch.save(model.state_dict(), ruta_last_model)
minutos_totales = (time.time() - inicio_entrenamiento) / 60
print(f'Entrenamiento finalizado en {minutos_totales:.2f} min. Mejor época: {best_epoch} (F1={mejor_f1_val:.4f})')

graficar_curvas_entrenamiento(history, best_epoch)

### Dinámica de Convergencia y Efecto de la Regularización

La ejecución de las 40 épocas completas permite verificar dos comportamientos complementarios. En primer lugar, liberar los bloques denseblock3 y denseblock4 con tasa de aprendizaje diferenciada (1e-4 en los filtros convolucionales y 1e-3 en el clasificador lineal) destraba la capacidad de representación de la red: mientras el modelo preliminar con bloques congelados se estancaba alrededor del 52% de exactitud desde las primeras iteraciones, la pérdida ponderada desciende de forma sostenida a medida que el agendador CosineAnnealingLR reduce el paso de actualización. En segundo lugar, la combinación de Dropout(p=0.35), decaimiento de pesos $ (1e-2), *label smoothing* ( .05) y aumentación afín moderada contiene la divergencia de la pérdida de validación en la segunda mitad del entrenamiento, resguardando en est_multiclase_model.pth el estado de mayor F1-Macro antes de pasar a la evaluación final.

---
## Parte 05: Evaluación en el Conjunto de Prueba y Desempeño por Etapas

Para evaluar los pesos guardados (`best_multiclase_model.pth`) sobre el conjunto de prueba (`249` imágenes de `217` pacientes no vistos durante el entrenamiento ni la validación), el bloque ajusta primero un factor de escala de temperatura $T > 0$ minimizando la pérdida logarítmica negativa sobre los *logits* de validación ($\hat{p}_i = \text{Softmax}(z_i / T)$). Con esas probabilidades calibradas, el código reporta en primer lugar el desempeño multiclase directo asignando cada placa a la categoría de mayor probabilidad ($\arg\max$).

A continuación, la evaluación desglosa el sistema en sus dos etapas operativas:
- **Etapa 1 — Triaje binario Sano vs. Enfermo:** la probabilidad de patología surge de la suma $P(\text{Enfermo}) = P(1) + P(2) = 1 - P(0)$. La versión inicial del proyecto buscó el umbral que maximizara el F1 binario; sin embargo, como en el problema binario el $56.2\%$ de las muestras son patológicas ($140$ enfermos frente a $109$ sanos en *Test*) y la fórmula del F1 omite los verdaderos negativos ($TN$), ese criterio desplazaba la especificidad a valores de entre $0.16$ y $0.26$. Para evitar ese sesgo, la rutina calcula sobre la curva ROC de validación dos puntos de corte alternativos: el **umbral de Youden** ($\max(\text{Sensibilidad} + \text{Especificidad} - 1)$), que equilibra ambas tasas, y un **umbral clínico de tamizaje** que exige una sensibilidad mínima del $90\%$ en validación.
- **Etapa 2 — Diferenciación entre Cardiomegalia y Derrame Pleural:** sobre las placas patológicas, el clasificador decide entre ambas afecciones a partir de la probabilidad condicional $P(\text{Derrame} \mid \text{Enfermo}) = \frac{P(2)}{P(1) + P(2)}$.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader

RUTA_MEJOR_CHECKPOINT = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/checkpoints/best_multiclase_model.pth'
)
ETIQUETAS_CLASE = ['Sano (0)', 'Cardiomegalia (1)', 'Derrame (2)']


@torch.no_grad()
def inferir_logits(red: nn.Module, loader: DataLoader, disp: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    red.eval()
    logits_acum, etiquetas_acum = [], []
    for imagenes, etiquetas in loader:
        logits_acum.append(red(imagenes.to(disp)).cpu())
        etiquetas_acum.append(etiquetas.cpu())
    return torch.cat(logits_acum, dim=0), torch.cat(etiquetas_acum, dim=0)


def ajustar_temperatura_validacion(logits_val: torch.Tensor, y_val: torch.Tensor) -> float:
    """Ajusta el escalar T > 0 por L-BFGS minimizando NLL en validación."""
    logits_estandar = logits_val.detach().clone()
    objetivos_estandar = y_val.detach().clone()
    temperatura = nn.Parameter(torch.ones(1))
    criterio_nll = nn.CrossEntropyLoss()
    optimizador_lbfgs = torch.optim.LBFGS([temperatura], lr=0.01, max_iter=50)

    def paso_evaluacion():
        optimizador_lbfgs.zero_grad()
        # Se acota a 0.05 para impedir pasos degenerados de L-BFGS hacia T <= 0.
        perdida = criterio_nll(logits_estandar / temperatura.clamp(min=0.05), objetivos_estandar)
        perdida.backward()
        return perdida

    optimizador_lbfgs.step(paso_evaluacion)
    return float(temperatura.clamp(min=0.05).item())


def calcular_umbrales_roc_triaje(
    y_val_bin: np.ndarray, probs_anormal_val: np.ndarray, sensibilidad_minima: float = 0.90
) -> tuple[float, float]:
    """Obtiene los cortes de Youden y de tamizaje clínico sobre la curva ROC de validación."""
    fpr, tpr, umbrales = roc_curve(y_val_bin, probs_anormal_val)
    especificidad = 1.0 - fpr

    corte_youden = float(umbrales[np.argmax(tpr + especificidad - 1.0)])
    candidatos_clinicos = np.where(tpr >= sensibilidad_minima)[0]
    if len(candidatos_clinicos) == 0:
        return corte_youden, corte_youden

    mejor_idx_clinico = candidatos_clinicos[np.argmax(especificidad[candidatos_clinicos])]
    return corte_youden, float(umbrales[mejor_idx_clinico])


def evaluar_regimen_triaje(
    y_bin: np.ndarray, probs_anormal: np.ndarray, umbral: float, nombre_regimen: str
) -> np.ndarray:
    pred_bin = (probs_anormal >= umbral).astype(int)
    exactitud = accuracy_score(y_bin, pred_bin)
    vpp, sensibilidad, f1, _ = precision_recall_fscore_support(
        y_bin, pred_bin, average='binary', zero_division=0
    )
    matriz = confusion_matrix(y_bin, pred_bin)
    tn, fp, fn, tp = matriz.ravel()
    especificidad = tn / (tn + fp) if (tn + fp) else 0.0
    auc_bin = roc_auc_score(y_bin, probs_anormal)

    print(f'\n[{nombre_regimen} | corte = {umbral:.2f}]')
    print(f'  Accuracy:      {exactitud:.4f} | F1: {f1:.4f} | ROC-AUC: {auc_bin:.4f}')
    print(f'  Sensibilidad:  {sensibilidad:.4f} ({tp}/{tp + fn} enfermos) | VPP: {vpp:.4f}')
    print(f'  Especificidad: {especificidad:.4f} ({tn}/{tn + fp} sanos)')
    return matriz


if isinstance(model.classifier, nn.Linear):
    dim_in = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.35), nn.Linear(dim_in, 3)).to(device)

model.load_state_dict(torch.load(RUTA_MEJOR_CHECKPOINT, map_location=device))
model.to(device)

val_logits_t, val_labels_t = inferir_logits(model, val_loader, device)
test_logits_t, test_labels_t = inferir_logits(model, test_loader, device)

T_opt = ajustar_temperatura_validacion(val_logits_t, val_labels_t)
print(f'Temperatura de calibración (Validación): T = {T_opt:.3f}')

val_probs = torch.softmax(val_logits_t / T_opt, dim=1).numpy()
val_labels = val_labels_t.numpy()
test_probs = torch.softmax(test_logits_t / T_opt, dim=1).numpy()
test_labels = test_labels_t.numpy()

preds_argmax = test_probs.argmax(axis=1)
acc_multi = accuracy_score(test_labels, preds_argmax)
auc_multi_ovr = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')

print(f'\n Rendimiento Multiclase Global (Test, n={len(test_labels)}) ')
print(f'Accuracy: {acc_multi:.4f} | ROC-AUC Macro (OvR): {auc_multi_ovr:.4f}')
print(classification_report(test_labels, preds_argmax, target_names=ETIQUETAS_CLASE, zero_division=0))

cm_multi = confusion_matrix(test_labels, preds_argmax)
fig, (ax_cm, ax_pr) = plt.subplots(1, 2, figsize=(13, 4.8))
sns.heatmap(
    cm_multi, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
    xticklabels=['Sano', 'Cardio', 'Derrame'], yticklabels=['Sano', 'Cardio', 'Derrame'],
)
ax_cm.set(title='Matriz de Confusión Multiclase (Argmax)', ylabel='Diagnóstico Real', xlabel='Predicción')

for idx, (nombre, col) in enumerate(zip(ETIQUETAS_CLASE, ['#2ca02c', '#1f77b4', '#d62728'])):
    binario_clase = (test_labels == idx).astype(int)
    prec_c, rec_c, _ = precision_recall_curve(binario_clase, test_probs[:, idx])
    ap_c = average_precision_score(binario_clase, test_probs[:, idx])
    ax_pr.plot(rec_c, prec_c, color=col, lw=2, label=f'{nombre} (AP={ap_c:.2f})')
ax_pr.set(title='Curvas Precision-Recall por Clase', xlabel='Sensibilidad', ylabel='Precisión')
ax_pr.legend()
ax_pr.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

val_labels_bin = (val_labels > 0).astype(int)
val_probs_anormal = val_probs[:, 1] + val_probs[:, 2]
umbral_youden, umbral_clinico = calcular_umbrales_roc_triaje(val_labels_bin, val_probs_anormal)
print(f'Umbrales en Validación -> Youden: {umbral_youden:.3f} | Clínico (Sens>=90%): {umbral_clinico:.3f}')

test_labels_bin = (test_labels > 0).astype(int)
test_probs_anormal = test_probs[:, 1] + test_probs[:, 2]

cm_std = evaluar_regimen_triaje(test_labels_bin, test_probs_anormal, 0.50, 'Etapa 1 - Estándar')
cm_youden = evaluar_regimen_triaje(test_labels_bin, test_probs_anormal, umbral_youden, 'Etapa 1 - Youden')
cm_clin = evaluar_regimen_triaje(test_labels_bin, test_probs_anormal, umbral_clinico, 'Etapa 1 - Clínico Sens>=90%')

fig, (ax_y, ax_c, ax_h) = plt.subplots(1, 3, figsize=(16, 4.4))
sns.heatmap(
    cm_youden, annot=True, fmt='d', cmap='Greens', ax=ax_y,
    xticklabels=['Sano (0)', 'Anormal (1)'], yticklabels=['Sano (0)', 'Anormal (1)'],
)
ax_y.set(title=f'Triaje Youden (>= {umbral_youden:.2f})', ylabel='Real', xlabel='Predicción')

sns.heatmap(
    cm_clin, annot=True, fmt='d', cmap='Blues', ax=ax_c,
    xticklabels=['Sano (0)', 'Anormal (1)'], yticklabels=['Sano (0)', 'Anormal (1)'],
)
ax_c.set(title=f'Triaje Clínico (>= {umbral_clinico:.2f})', ylabel='Real', xlabel='Predicción')

ax_h.hist(test_probs_anormal[test_labels_bin == 0], bins=20, alpha=0.6, color='green', label='Sano (0)', density=True)
ax_h.hist(test_probs_anormal[test_labels_bin == 1], bins=20, alpha=0.6, color='red', label='Enfermo (1+2)', density=True)
ax_h.axvline(umbral_youden, color='darkgreen', ls='--', lw=2, label=f'Youden ({umbral_youden:.2f})')
ax_h.axvline(umbral_clinico, color='navy', ls=':', lw=2, label=f'Clínico ({umbral_clinico:.2f})')
ax_h.set(title='Densidad de P(Enfermo)', xlabel='P(Cardio) + P(Derrame)', ylabel='Densidad')
ax_h.legend()
plt.tight_layout()
plt.show()

mask_enfermos = test_labels > 0
labels_etapa2 = (test_labels[mask_enfermos] == 2).astype(int)
probs_cardio = test_probs[mask_enfermos, 1]
probs_derrame = test_probs[mask_enfermos, 2]
probs_cond_derrame = probs_derrame / np.clip(probs_cardio + probs_derrame, 1e-8, None)
preds_etapa2 = (probs_cond_derrame >= 0.50).astype(int)

auc_etapa2 = roc_auc_score(labels_etapa2, probs_cond_derrame)
print(f'\n[Etapa 2 - Diferencial Cardiomegalia vs. Derrame | ROC-AUC = {auc_etapa2:.4f}]')
print(classification_report(
    labels_etapa2, preds_etapa2, target_names=['Cardiomegalia (1)', 'Derrame Pleural (2)'], zero_division=0
))

Temperatura de calibración (Validación): T = 1.121

 Rendimiento Multiclase Global (Test, n=249) 
Accuracy: 0.6908 | ROC-AUC Macro (OvR): 0.8654
                   precision    recall  f1-score   support

         Sano (0)       0.69      0.72      0.70       109
Cardiomegalia (1)       0.81      0.50      0.62        58
      Derrame (2)       0.65      0.79      0.71        82

         accuracy                           0.69       249
        macro avg       0.72      0.67      0.68       249
     weighted avg       0.70      0.69      0.69       249
Umbrales en Validación -> Youden: 0.578 | Clínico (Sens>=90%): 0.353

[Etapa 1 - Estándar | corte = 0.50]
  Accuracy:      0.7430 | F1: 0.7714 | ROC-AUC: 0.8231
  Sensibilidad:  0.7714 (108/140 enfermos) | VPP: 0.7714
  Especificidad: 0.7064 (77/109 sanos)

[Etapa 1 - Youden | corte = 0.58]
  Accuracy:      0.7390 | F1: 0.7636 | ROC-AUC: 0.8231
  Sensibilidad:  0.7500 (105/140 enfermos) | VPP: 0.7778
  Especificidad: 0.7248 (79/109 sano

### Visualización de Resultados y Exportación de Figuras para Exposición

El siguiente bloque reúne cinco funciones modulares diseñadas para representar gráficamente el comportamiento del sistema en el conjunto de prueba (`n = 249`) y exportar cada panel en alta resolución (`300 DPI`) dentro de la carpeta `figuras_presentacion/`:

1. **`graficar_comparativa_evolucion_modelo()`**: contrasta el ensayo preliminar (red congelada y catálogo sin depurar) frente al modelo final con ajuste fino y curación de etiquetas, tanto en métricas multiclase como en el balance entre sensibilidad y especificidad de la Etapa 1.
2. **`graficar_analisis_multiclase_completo()`**: integra la matriz de confusión normalizada por fila (porcentaje de acierto sobre el total real de cada diagnóstico), las curvas ROC *One-vs-Rest* de las tres categorías y el desglose de precisión, sensibilidad y F1 por clase.
3. **`graficar_triaje_etapa1_detallado()`**: grafica la curva ROC del triaje binario (`ROC-AUC = 0.8231`) señalando la posición de los tres umbrales (`0.35`, `0.50` y `0.58`), la variación continua de sensibilidad, especificidad e índice J de Youden a lo largo de todo el rango de corte y el conteo de aciertos y errores clínicos (`TP`, `TN`, `FP`, `FN`).
4. **`graficar_etapa2_diferencial()`**: presenta la matriz de confusión diferencial entre Cardiomegalia y Derrame Pleural (`n = 140`), su curva ROC (`ROC-AUC = 0.9521`) junto a la curva Precisión-Sensibilidad y la separación de densidades de la probabilidad condicional $P(\text{Derrame} \mid \text{Enfermo})$.
5. **`graficar_galeria_predicciones_clinicas()`**: extrae placas representativas del conjunto de prueba y muestra sobre cada radiografía el diagnóstico real junto con las probabilidades asignadas en la Etapa 1 y en la Etapa 2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    auc,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_curve,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 11.5,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.labelweight': 'bold',
})

RUTA_DATASET_DRIVE = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset'
)
DIR_FIGURAS = (RUTA_DATASET_DRIVE / 'figuras_presentacion') if RUTA_DATASET_DRIVE.exists() else Path('./figuras_presentacion')
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)


def _rotular_barras(ax: plt.Axes, grupos_barras, formato: str = '{:.2f}', escala: float = 1.0, size: float = 8.5) -> None:
    for grupo in grupos_barras:
        for barra in grupo:
            altura = barra.get_height()
            ax.annotate(
                formato.format(altura * escala),
                xy=(barra.get_x() + barra.get_width() / 2, altura),
                xytext=(0, 3),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=size,
                fontweight='bold',
            )


def _matriz_con_porcentajes(y_real: np.ndarray, y_pred: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    conteos = confusion_matrix(y_real, y_pred)
    porcentajes = conteos.astype(float) / conteos.sum(axis=1, keepdims=True) * 100.0
    etiquetas = np.array(
        [[f'{conteos[i, j]}\n({porcentajes[i, j]:.1f}%)' for j in range(conteos.shape[1])] for i in range(conteos.shape[0])],
        dtype=object,
    )
    return porcentajes, etiquetas


def graficar_comparativa_evolucion_modelo(dir_salida: Path = DIR_FIGURAS) -> Path:
    """Contrasta el prototipo inicial congelado frente al modelo con ajuste fino y curación NLP."""
    fig, (ax_multi, ax_triaje) = plt.subplots(1, 2, figsize=(16, 5.6))

    metricas = ['Accuracy\nGlobal', 'F1-Score\nMacro', 'F1 Sano\n(0)', 'F1 Cardio\n(1)', 'F1 Derrame\n(2)']
    base_congelada = [0.5217, 0.5100, 0.4700, 0.4700, 0.5900]
    modelo_final = [0.6908, 0.6800, 0.7000, 0.6200, 0.7100]

    pos = np.arange(len(metricas))
    ancho = 0.36
    b_ini = ax_multi.bar(pos - ancho / 2, base_congelada, ancho, label='Inicial (Congelado / sin depurar)', color='#94a3b8', edgecolor='#334155')
    b_fin = ax_multi.bar(pos + ancho / 2, modelo_final, ancho, label='Final (Fine-Tuning / CSV depurado)', color='#0284c7', edgecolor='#0f172a')
    ax_multi.set(ylim=(0.0, 1.02), ylabel='Puntaje (0.0 a 1.0)', title='a) Evolución Multiclase (Test, n = 249)', xticks=pos, xticklabels=metricas)
    ax_multi.legend(loc='upper left')
    ax_multi.grid(axis='y', ls='--', alpha=0.5)
    _rotular_barras(ax_multi, [b_ini, b_fin])

    cortes = ['Inicial\n(0.50)', 'Inicial\n(0.57)', 'Clínico\n(0.35)', 'Estándar\n(0.50)', 'Youden\n(0.58)']
    sens = [0.9790, 0.9590, 0.8357, 0.7714, 0.7500]
    esp = [0.1600, 0.2640, 0.6422, 0.7064, 0.7248]
    acc = [0.7010, 0.7230, 0.7510, 0.7430, 0.7390]

    pos_t = np.arange(len(cortes))
    w_t = 0.26
    b_s = ax_triaje.bar(pos_t - w_t, sens, w_t, label='Sensibilidad (Enfermos)', color='#0d9488', edgecolor='#0f172a')
    b_e = ax_triaje.bar(pos_t, esp, w_t, label='Especificidad (Sanos)', color='#e11d48', edgecolor='#0f172a')
    b_a = ax_triaje.bar(pos_t + w_t, acc, w_t, label='Accuracy Triaje', color='#1e293b', edgecolor='#0f172a')
    ax_triaje.set(ylim=(0.0, 1.12), ylabel='Proporción', title='b) Etapa 1: Recuperación de la Especificidad', xticks=pos_t, xticklabels=cortes)
    ax_triaje.legend(loc='upper right')
    ax_triaje.grid(axis='y', ls='--', alpha=0.5)
    _rotular_barras(ax_triaje, [b_s, b_e, b_a], formato='{:.0f}%', escala=100.0, size=8.0)

    plt.tight_layout()
    ruta_png = dir_salida / '01_comparativa_evolucion_modelo.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_analisis_multiclase_completo(
    y_real: np.ndarray, probs: np.ndarray, dir_salida: Path = DIR_FIGURAS
) -> Path:
    nombres = ['Sano (0)', 'Cardiomegalia (1)', 'Derrame (2)']
    colores = ['#16a34a', '#0284c7', '#dc2626']
    pred = probs.argmax(axis=1)
    pct_fila, anotaciones = _matriz_con_porcentajes(y_real, pred)

    fig, (ax_cm, ax_roc, ax_bar) = plt.subplots(1, 3, figsize=(18, 5.3))

    sns.heatmap(
        pct_fila, annot=anotaciones, fmt='', cmap='Blues', vmin=0, vmax=100,
        xticklabels=['Sano (0)', 'Cardio (1)', 'Derrame (2)'],
        yticklabels=['Sano (0)', 'Cardio (1)', 'Derrame (2)'],
        annot_kws={'size': 10, 'weight': 'bold'}, ax=ax_cm,
    )
    ax_cm.set(title='a) Matriz Multiclase (Conteo y % por Fila)', ylabel='Diagnóstico Real', xlabel='Predicción (Argmax)')

    fpr_unificado = np.unique(np.concatenate([roc_curve((y_real == i).astype(int), probs[:, i])[0] for i in range(3)]))
    tpr_promedio = np.zeros_like(fpr_unificado)
    for i, (nombre, col) in enumerate(zip(nombres, colores)):
        fpr_i, tpr_i, _ = roc_curve((y_real == i).astype(int), probs[:, i])
        tpr_promedio += np.interp(fpr_unificado, fpr_i, tpr_i)
        ax_roc.plot(fpr_i, tpr_i, color=col, lw=2.2, label=f'{nombre} (AUC = {auc(fpr_i, tpr_i):.3f})')

    tpr_promedio /= 3.0
    ax_roc.plot(fpr_unificado, tpr_promedio, color='#1e1b4b', ls='--', lw=2.5, label=f'Macro OvR (AUC = {auc(fpr_unificado, tpr_promedio):.4f})')
    ax_roc.plot([0, 1], [0, 1], color='gray', ls=':', lw=1.2)
    ax_roc.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad', ylabel='Sensibilidad', title='b) Curvas ROC One-vs-Rest')
    ax_roc.legend(loc='lower right')
    ax_roc.grid(True, ls='--', alpha=0.5)

    prec_c, rec_c, f1_c, sop_c = precision_recall_fscore_support(y_real, pred, labels=[0, 1, 2], zero_division=0)
    pos = np.arange(3)
    w = 0.25
    b_p = ax_bar.bar(pos - w, prec_c, w, label='Precisión (VPP)', color='#0284c7', edgecolor='#0f172a')
    b_r = ax_bar.bar(pos, rec_c, w, label='Sensibilidad', color='#0d9488', edgecolor='#0f172a')
    b_f = ax_bar.bar(pos + w, f1_c, w, label='F1-Score', color='#4f46e5', edgecolor='#0f172a')
    ax_bar.set(
        ylim=(0.0, 1.08), xticks=pos, xticklabels=[f'{nombres[i]}\n(n = {sop_c[i]})' for i in range(3)],
        ylabel='Puntaje', title='c) Métricas por Clase en Test',
    )
    ax_bar.legend(loc='upper left')
    ax_bar.grid(axis='y', ls='--', alpha=0.5)
    _rotular_barras(ax_bar, [b_p, b_r, b_f])

    plt.tight_layout()
    ruta_png = dir_salida / '02_analisis_multiclase_completo.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_triaje_etapa1_detallado(
    y_bin: np.ndarray, probs_anormal: np.ndarray, u_youden: float, u_clinico: float, dir_salida: Path = DIR_FIGURAS
) -> Path:
    fpr, tpr, _ = roc_curve(y_bin, probs_anormal)
    fig, (ax_roc, ax_umbral, ax_conteos) = plt.subplots(1, 3, figsize=(18, 5.3))

    ax_roc.plot(fpr, tpr, color='#0f172a', lw=2.5, label=f'ROC Etapa 1 (AUC = {auc(fpr, tpr):.4f})')
    ax_roc.fill_between(fpr, tpr, alpha=0.12, color='#0284c7')
    ax_roc.plot([0, 1], [0, 1], color='gray', ls=':')

    puntos_operativos = [
        ('Estándar (0.50)', 0.50, '#f59e0b', 'o'),
        (f'Clínico ({u_clinico:.2f})', u_clinico, '#0284c7', 's'),
        (f'Youden ({u_youden:.2f})', u_youden, '#16a34a', 'D'),
    ]
    for nombre_u, valor_u, col, marcador in puntos_operativos:
        tn, fp, fn, tp = confusion_matrix(y_bin, (probs_anormal >= valor_u).astype(int)).ravel()
        sens_u, fpr_u = tp / (tp + fn), fp / (fp + tn)
        ax_roc.scatter(
            [fpr_u], [sens_u], color=col, s=95, marker=marcador, zorder=5, edgecolor='black',
            label=f'{nombre_u}: Sens={sens_u*100:.1f}%, Esp={(1-fpr_u)*100:.1f}%',
        )
    ax_roc.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad', ylabel='Sensibilidad', title='a) Curva ROC Etapa 1 y Umbrales')
    ax_roc.legend(loc='lower right', fontsize=8.5)
    ax_roc.grid(True, ls='--', alpha=0.5)

    grilla_u = np.linspace(0.05, 0.95, 181)
    tasas = np.array([
        confusion_matrix(y_bin, (probs_anormal >= u).astype(int)).ravel() for u in grilla_u
    ])
    tn_g, fp_g, fn_g, tp_g = tasas[:, 0], tasas[:, 1], tasas[:, 2], tasas[:, 3]
    curva_sens = tp_g / (tp_g + fn_g)
    curva_esp = tn_g / (tn_g + fp_g)

    ax_umbral.plot(grilla_u, curva_sens, color='#dc2626', lw=2.2, label='Sensibilidad')
    ax_umbral.plot(grilla_u, curva_esp, color='#16a34a', lw=2.2, label='Especificidad')
    ax_umbral.plot(grilla_u, curva_sens + curva_esp - 1.0, color='#4f46e5', ls='-.', lw=2.0, label='Índice J de Youden')
    ax_umbral.axvline(u_clinico, color='#0284c7', ls=':', lw=2.0, label=f'Clínico ({u_clinico:.2f})')
    ax_umbral.axvline(u_youden, color='#15803d', ls='--', lw=2.0, label=f'Youden ({u_youden:.2f})')
    ax_umbral.set(xlim=(0.05, 0.95), ylim=(0.0, 1.05), xlabel='Umbral sobre P(Enfermo)', ylabel='Proporción', title='b) Sensibilidad vs. Especificidad')
    ax_umbral.legend(loc='lower left', fontsize=8.5)
    ax_umbral.grid(True, ls='--', alpha=0.5)

    etiquetas_u = ['Estándar\n(0.50)', f'Clínico\n({u_clinico:.2f})', f'Youden\n({u_youden:.2f})']
    desglose = np.array([confusion_matrix(y_bin, (probs_anormal >= u).astype(int)).ravel() for u in (0.50, u_clinico, u_youden)])
    pos_c = np.arange(3)
    w_c = 0.20
    b_tp = ax_conteos.bar(pos_c - 1.5 * w_c, desglose[:, 3], w_c, label='Enfermos Detectados (TP)', color='#0d9488', edgecolor='black')
    b_tn = ax_conteos.bar(pos_c - 0.5 * w_c, desglose[:, 0], w_c, label='Sanos Descartados (TN)', color='#16a34a', edgecolor='black')
    b_fp = ax_conteos.bar(pos_c + 0.5 * w_c, desglose[:, 1], w_c, label='Falsas Alarmas (FP)', color='#f59e0b', edgecolor='black')
    b_fn = ax_conteos.bar(pos_c + 1.5 * w_c, desglose[:, 2], w_c, label='No Detectados (FN)', color='#dc2626', edgecolor='black')
    ax_conteos.set(ylim=(0, 155), xticks=pos_c, xticklabels=etiquetas_u, ylabel='Pacientes (n = 249)', title='c) Conteo Clínico por Punto de Corte')
    ax_conteos.legend(loc='upper right', fontsize=8.5)
    ax_conteos.grid(axis='y', ls='--', alpha=0.5)
    _rotular_barras(ax_conteos, [b_tp, b_tn, b_fp, b_fn], formato='{:.0f}')

    plt.tight_layout()
    ruta_png = dir_salida / '03_triaje_etapa1_detallado.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_etapa2_diferencial(
    y_etapa2: np.ndarray, probs_cond: np.ndarray, pred_etapa2: np.ndarray, dir_salida: Path = DIR_FIGURAS
) -> Path:
    pct_fila, anotaciones = _matriz_con_porcentajes(y_etapa2, pred_etapa2)
    fig, (ax_cm, ax_curvas, ax_hist) = plt.subplots(1, 3, figsize=(18, 5.1))

    sns.heatmap(
        pct_fila, annot=anotaciones, fmt='', cmap='YlGnBu', vmin=0, vmax=100,
        xticklabels=['Cardiomegalia (1)', 'Derrame Pleural (2)'],
        yticklabels=['Cardiomegalia (1)', 'Derrame Pleural (2)'],
        annot_kws={'size': 11, 'weight': 'bold'}, ax=ax_cm,
    )
    ax_cm.set(title='a) Matriz Etapa 2 (n = 140 Enfermos)', ylabel='Patología Real', xlabel='Predicción (Corte 0.50)')

    fpr2, tpr2, _ = roc_curve(y_etapa2, probs_cond)
    prec2, rec2, _ = precision_recall_curve(y_etapa2, probs_cond)
    ax_curvas.plot(fpr2, tpr2, color='#0d9488', lw=2.5, label=f'ROC Diferencial (AUC = {auc(fpr2, tpr2):.4f})')
    ax_curvas.plot(rec2, prec2, color='#4f46e5', ls='--', lw=2.2, label=f'Precision-Recall (AP = {average_precision_score(y_etapa2, probs_cond):.4f})')
    ax_curvas.plot([0, 1], [0, 1], color='gray', ls=':')
    ax_curvas.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad / Recall', ylabel='Sensibilidad / Precisión', title='b) Discriminación en Etapa 2')
    ax_curvas.legend(loc='lower left')
    ax_curvas.grid(True, ls='--', alpha=0.5)

    ax_hist.hist(probs_cond[y_etapa2 == 0], bins=18, alpha=0.65, color='#0284c7', edgecolor='black', density=True, label='Cardiomegalia (n=58)')
    ax_hist.hist(probs_cond[y_etapa2 == 1], bins=18, alpha=0.65, color='#dc2626', edgecolor='black', density=True, label='Derrame Pleural (n=82)')
    ax_hist.axvline(0.50, color='#0f172a', ls='--', lw=2.2, label='Corte Diferencial (0.50)')
    ax_hist.set(xlim=(0.0, 1.0), xlabel='P(Derrame | Enfermo)', ylabel='Densidad', title='c) Separación de Probabilidades')
    ax_hist.legend(loc='upper center')
    ax_hist.grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    ruta_png = dir_salida / '04_etapa2_diagnostico_diferencial.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_galeria_predicciones_clinicas(
    tabla_test: pd.DataFrame | None, y_real: np.ndarray, probs: np.ndarray, u_clinico: float, dir_salida: Path = DIR_FIGURAS
) -> Path | None:
    if tabla_test is None or 'full_path' not in tabla_test.columns:
        return None

    # TODO: Superponer mapas de activación Grad-CAM de denseblock4 sobre cada placa
    # para auditar visualmente si la atención recae en la silueta cardíaca o el seno costofrénico.
    df_vis = tabla_test.reset_index(drop=True)
    nombres = {0: 'Sano (0)', 1: 'Cardiomegalia (1)', 2: 'Derrame Pleural (2)'}
    seleccion = []

    pred_argmax = probs.argmax(axis=1)
    for clase in (0, 1, 2):
        aciertos_clase = np.where((y_real == clase) & (pred_argmax == clase))[0]
        fuente = aciertos_clase if len(aciertos_clase) >= 2 else np.where(y_real == clase)[0]
        orden_confianza = fuente[np.argsort(probs[fuente, clase])[::-1]]
        seleccion.extend(orden_confianza[:2])

    fig, axes = plt.subplots(2, 3, figsize=(15, 9.2))
    for ax, idx_fila in zip(axes.ravel(), seleccion[:6]):
        ruta_img = Path(df_vis.loc[idx_fila, 'full_path'])
        clase_real = int(y_real[idx_fila])
        _, p_cardio, p_derrame = probs[idx_fila]
        p_enfermo = p_cardio + p_derrame

        if p_enfermo < u_clinico:
            diagnostico_sistema = 'Sano (Descarte Etapa 1)'
            texto_etapa2 = 'Sin pase a Etapa 2'
        else:
            p_cond = p_derrame / max(p_enfermo, 1e-8)
            diagnostico_sistema = 'Derrame Pleural (Etapa 2)' if p_cond >= 0.50 else 'Cardiomegalia (Etapa 2)'
            texto_etapa2 = f'P(Derr|Enf): {p_cond*100:.1f}%'

        if ruta_img.exists():
            with Image.open(ruta_img) as img:
                ax.imshow(img.convert('L'), cmap='bone')
        ax.axis('off')
        color_rotulo = '#15803d' if pred_argmax[idx_fila] == clase_real else '#dc2626'
        ax.set_title(
            f'Real: {nombres[clase_real]}\n{diagnostico_sistema}\nEtapa 1 P(Enf): {p_enfermo*100:.1f}% | {texto_etapa2}',
            fontsize=9.5, color=color_rotulo, fontweight='bold',
        )

    plt.suptitle('Inferencia en Dos Etapas sobre Radiografías del Conjunto de Prueba', fontsize=13.5, fontweight='bold', y=1.01)
    plt.tight_layout()
    ruta_png = dir_salida / '05_galeria_predicciones_clinicas.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


graficar_comparativa_evolucion_modelo()
if 'test_labels' in globals() and 'test_probs' in globals():
    graficar_analisis_multiclase_completo(test_labels, test_probs)
    graficar_triaje_etapa1_detallado(test_labels_bin, test_probs_anormal, umbral_youden, umbral_clinico)
    graficar_etapa2_diferencial(labels_etapa2, probs_cond_derrame, preds_etapa2)
    graficar_galeria_predicciones_clinicas(globals().get('df_test'), test_labels, test_probs, umbral_clinico)

### Discusión de los Resultados en el Conjunto de Prueba

En la evaluación multiclase directa (Fase A), la exactitud global subió del `52.17%` obtenido con la red congelada al **`69.08%`** (`+16.91 puntos`), acompañada de un F1-Macro de **`0.6800`** y un área bajo la curva ROC (One-vs-Rest) de **`0.8654`**. El análisis por categoría muestra tres mejoras complementarias: `Sano (0)` duplicó su sensibilidad pasando de `0.36` a **`0.72`** con una precisión de **`0.69`** (F1 = **`0.70`**); `Derrame Pleural (2)` elevó su precisión de `0.46` a **`0.65`** conservando un *recall* de **`0.79`** (F1 = **`0.71`**), lo que confirma que dejó de actuar como clase sumidero; y `Cardiomegalia (1)` alcanzó la precisión más alta del conjunto (**`0.81`** frente al `0.54` inicial) con un *recall* de **`0.50`** y un F1 de **`0.62`**.

Para el triaje binario de la Etapa 1 (Fase B), la calibración por temperatura convergió en $T = 1.121$ y el área bajo la curva ROC pasó de `0.7254` a **`0.8231`**. Al evaluar los tres puntos operativos sobre el conjunto de prueba (`140` enfermos y `109` sanos), el **umbral clínico de tamizaje (`0.353` $\approx 0.35$)** logra la mayor exactitud global del triaje (**`75.10%`**) y el mayor F1 binario (**`0.7905`**), detectando el **`83.57%`** de los casos patológicos ($117/140$) con un VPP de **`75.00%`** y una especificidad del **`64.22%`** ($70/109$ sanos descartados, cuatro veces superior al `16.0%` inicial). Con el **umbral estándar (`0.50`)**, la exactitud se sitúa en **`74.30%`** (F1 = **`0.7714`**), equilibrando una sensibilidad del **`77.14%`** ($108/140$) con una especificidad del **`70.64%`** ($77/109$). Por su parte, el **umbral de Youden (`0.578` $\approx 0.58$)** maximiza el descarte de estudios normales llevando la especificidad al **`72.48%`** ($79/109$) y el VPP al **`77.78%`**, con una sensibilidad del **`75.00%`** ($105/140$) y una exactitud del **`73.90%`**.

Por último, la Etapa 2 (Fase C) confirma la utilidad de separar el sistema en dos etapas. Al restringir la comparación a los `140` casos patológicos mediante la probabilidad condicional $P(\text{Derrame} \mid \text{Enfermo})$, la discriminación entre Cardiomegalia y Derrame Pleural alcanza un **ROC-AUC de `0.9521`** y una exactitud del **`88.00%`** (F1-Macro = **`0.8700`**): `Cardiomegalia (1)` registra una precisión de **`0.92`**, *recall* de **`0.78`** y F1 de **`0.84`**, mientras que `Derrame Pleural (2)` obtiene una precisión de **`0.86`**, *recall* de **`0.95`** y F1 de **`0.90`**. La diferencia entre el `88.00%` de exactitud en la Etapa 2 y el `69.08%` de la clasificación conjunta de tres vías indica que la red distingue con claridad ambas patologías entre sí, y que los errores del clasificador multiclase se concentran en cardiomegalias leves que se superponen con tórax normales.

---
## Conclusiones Finales

| Etapa / Dimensión | Métrica | Modelo Inicial (Red Congelada / CSV sin depurar) | Modelo Final (Fine-Tuning / CSV depurado) | Diferencia |
| :--- | :--- | :---: | :---: | :---: |
| **Multiclase (3 vías)** | Accuracy Global | `52.17%` | **`69.08%`** | `+16.91 pts` |
| **Multiclase (3 vías)** | F1-Score Macro | `0.5100` | **`0.6800`** | `+0.1700` |
| **Multiclase (3 vías)** | ROC-AUC Macro (OvR) | — | **`0.8654`** | — |
| **Por clase** | F1 `Sano (0)` | `0.47` (Recall `0.36`) | **`0.70` (Recall `0.72`)** | `+0.23` |
| **Por clase** | F1 `Cardiomegalia (1)` | `0.47` (Precisión `0.54`) | **`0.62` (Precisión `0.81`)** | `+0.15` |
| **Por clase** | F1 `Derrame Pleural (2)` | `0.59` (Precisión `0.46`) | **`0.71` (Precisión `0.65`)** | `+0.12` |
| **Etapa 1: Triaje** | ROC-AUC Binario | `0.7254` | **`0.8231`** | `+0.0977` |
| **Etapa 1: Triaje** | Especificidad (Youden `0.58`) | `26.40%` (umbral `0.57`) | **`72.48%`** (Recall `75.00%`, Acc `73.90%`) | `+46.08 pts` |
| **Etapa 1: Triaje** | Especificidad (Estándar `0.50`) | `16.00%` (umbral `0.50`) | **`70.64%`** (Recall `77.14%`, Acc `74.30%`) | `+54.64 pts` |
| **Etapa 1: Triaje** | Especificidad (Tamizaje `0.35`) | `16.00%` (umbral `0.50`) | **`64.22%`** (Recall `83.57%`, Acc `75.10%`) | `+48.22 pts` |
| **Etapa 2: Diferencial** | ROC-AUC (Cardio vs. Derrame) | — | **`0.9521` (Accuracy `88.00%`, F1 `0.87`)** | — |

Los resultados obtenidos a lo largo del trabajo permiten extraer cuatro conclusiones principales:

1. **Curación de etiquetas sobre catálogos generados por NLP:** En bases hospitalarias como *ChestX-ray8/14* (Wang et al., 2017), filtrar una patología buscando únicamente su nombre en la cadena `Finding Labels` arrastra cuadros combinados propios de la clínica torácica (Litjens et al., 2017). Retirar las placas donde coexistían `Cardiomegaly` y `Effusion` corrigió el sesgo que llevaba al clasificador inicial a predecir derrame pleural en más de la mitad del conjunto de prueba, elevando la precisión de `Cardiomegalia` a `0.81` y la de `Derrame Pleural` a `0.65`.
2. **Necesidad de adaptar las capas convolucionales profundas:** Congelar todos los bloques de `DenseNet-121` (Huang et al., 2017) impidió que el modelo aprendiera rasgos anatómicos específicos del tórax, limitándolo a correlaciones superficiales de brillo. Liberar los bloques `denseblock3` y `denseblock4` en PyTorch (Paszke et al., 2019) con una tasa de aprendizaje menor (`1e-4`) que la del cabezal lineal (`1e-3`), manteniendo una corrida completa de 40 épocas regulada mediante `Dropout`, penalización $L_2$ y *label smoothing*, permitió elevar la exactitud multiclase al `69.08%` (`ROC-AUC = 0.8654`) sin degradar su generalización en pacientes nuevos.
3. **Restricciones anatómicas en el preprocesamiento:** Suprimir el filtro de mediana previo al reescalado y eliminar la inversión horizontal (`RandomHorizontalFlip`) durante el entrenamiento evitó distorsionar la orientación izquierda de la silueta cardíaca, duplicando la sensibilidad de la categoría `Sano` (`0.36 -> 0.72`).
4. **Elección del criterio de umbral en esquemas en cascada:** Cuando se agrupan clases patológicas para un triaje binario a partir de una salida *Softmax* multiclase, maximizar el F1 binario sin contemplar los verdaderos negativos hunde la especificidad (`16% - 26%`). Calibrar la temperatura en validación ($T = 1.121$) y fijar el punto operativo sobre la curva ROC con *Scikit-learn* (Pedregosa et al., 2011) —ya sea mediante el umbral clínico en `0.35` (`Accuracy = 75.10%`, `Sensibilidad = 83.57%`, `Especificidad = 64.22%`), el corte estándar en `0.50` (`Especificidad = 70.64%`) o el índice J de Youden en `0.58` (`Especificidad = 72.48%`)— triplicó la capacidad de descartar tórax sanos y habilitó una segunda etapa diferencial con un área bajo la curva de **`0.9521`** (`Accuracy = 88.00%`).



## Referencias Bibliográficas

- Huang, G., Liu, Z., Van Der Maaten, L., & Weinberger, K. Q. (2017). Densely connected convolutional networks. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 4700–4708).
- Litjens, G., Kooi, T., Bejnordi, B. E., Setio, A. A. A., Ciompi, F., Ghafoorian, M., ... & Sánchez, C. I. (2017). A survey on deep learning in medical image analysis. *Medical Image Analysis*, 42, 60–88. https://doi.org/10.1016/j.media.2017.07.005
- Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., ... & Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. En *Advances in Neural Information Processing Systems (NeurIPS)* (Vol. 32).
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12(Oct), 2825–2830.
- Wang, X., Peng, Y., Lu, L., Lu, Z., Bagheri, M., & Summers, R. M. (2017). ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 2097–2106).